# Business Formation Analysis Project

This notebook is a starting point for exploring business formation data in the USA. The goal is to create a ML model that predicts next week state level business formation. Based on past 12 months of that state data AND 12 months national formation data including up to current week (ie the national level data of current week is used with past data on state level to predict unknown: current week at state level). Additionally some exploratory statistics of the data are made including graphs and the secondary aim is to identify if some of the features are more useful to predict the state level current week applications counts.

## Questions

- Can current week state level business formation be predicted accurately?
- Which features have the highest explanatory power for current week state level application.

## Data and analysis

The data source is Business Formation Statistics (BFS) from the U.S. Census Bureau at: www.census.gov

Two .csv datasets are used:
- bfs_us_apps_weekly_nsa.csv
- bfs_state_apps_weekly_nsa.csv

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

#pd.set_option("display.max_columns", 30)
#pd.set_option("display.width", 140)

TIME_KEYS = ["Year", "Week"]

for path in (US_FILE, STATE_FILE):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input not found: {path}. Place both CSV files beside the notebook."
        )

us_raw = pd.read_csv("bfs_us_apps_weekly_nsa.csv")
state_raw = pd.read_csv("bfs_state_apps_weekly_nsa.csv")

us_raw.head()

state_raw.head()

## Reshape state data and merge national features

In [ ]:
# Validate and normalize the time keys before reshaping or joining.
required_state = set(TIME_KEYS + ["State"])
required_us = set(TIME_KEYS)
if not required_state.issubset(state_raw.columns):
    raise ValueError(f"State file is missing required columns: {sorted(required_state - set(state_raw.columns))}")
if not required_us.issubset(us_raw.columns):
    raise ValueError(f"U.S. file is missing required columns: {sorted(required_us - set(us_raw.columns))}")

state_data = state_raw.copy()
us_data = us_raw.copy()
for frame in (state_data, us_data):
    for key in TIME_KEYS:
        frame[key] = pd.to_numeric(frame[key], errors="raise").astype("int64")

state_data["State"] = state_data["State"].astype("string").str.strip()
if state_data["State"].isna().any() or state_data["State"].eq("").any():
    raise ValueError("State file contains a missing or blank State value.")

state_features = [column for column in state_data.columns if column not in TIME_KEYS + ["State"]]
us_features = [column for column in us_data.columns if column not in TIME_KEYS]
if not state_features or not us_features:
    raise ValueError("Expected at least one non-time feature in each source file.")

for column in state_features:
    state_data[column] = pd.to_numeric(state_data[column], errors="raise")
for column in us_features:
    us_data[column] = pd.to_numeric(us_data[column], errors="raise")

state_key = TIME_KEYS + ["State"]
if state_data.duplicated(state_key).any():
    examples = state_data.loc[state_data.duplicated(state_key, keep=False), state_key].head()
    raise ValueError(f"State data has duplicate state-week records. Examples:\n{examples}")
if us_data.duplicated(TIME_KEYS).any():
    examples = us_data.loc[us_data.duplicated(TIME_KEYS, keep=False), TIME_KEYS].head()
    raise ValueError(f"U.S. data has duplicate week records. Examples:\n{examples}")

# pivot produces (feature, State) column pairs; flip that order for State_feature names.
state_wide = state_data.pivot(index=TIME_KEYS, columns="State", values=state_features)
state_wide.columns = [f"{state}_{feature}" for feature, state in state_wide.columns]
state_wide = state_wide.reset_index()

us_prefixed = us_data.rename(columns={column: f"USA_{column}" for column in us_features})

# Outer join preserves every observed week from either source; unmatched weeks remain visible as missing values.
combined = state_wide.merge(
    us_prefixed,
    on=TIME_KEYS,
    how="outer",
    validate="one_to_one",
    indicator="_source_match",
).sort_values(TIME_KEYS).reset_index(drop=True)

source_match_counts = combined["_source_match"].value_counts().reindex(
    ["both", "left_only", "right_only"], fill_value=0
)
combined = combined.drop(columns="_source_match")

#print(f"State-wide data: {state_wide.shape[0]:,} weeks x {state_wide.shape[1]} columns")
#print(f"Merged data: {combined.shape[0]:,} weeks x {combined.shape[1]} columns")
#print("Week coverage (both / state only / U.S. only):", source_match_counts.to_dict())
#print("States:", state_data["State"].nunique())
#first_week = combined.iloc[0]
#last_week = combined.iloc[-1]
#print(f"Time-key range: {int(first_week['Year'])} week {int(first_week['Week'])} "
#      f"to {int(last_week['Year'])} week {int(last_week['Week'])}")
#display(combined.head())

## Weekly application trends

The plots use `BA_NSA` as an initial headline measure. Counts are not seasonally adjusted, so recurring within-year patterns and differences in state size should be kept in mind. The state chart shows up to five states with the highest observed count in the latest available state week; this is a descriptive selection, not a representative sample.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)

if "USA_BA_NSA" in combined.columns:
    national = combined.dropna(subset=["USA_BA_NSA"]).copy()
    national["Time"] = national["Year"].astype(str) + "-W" + national["Week"].astype(str).str.zfill(2)
    axes[0].plot(national["Time"], national["USA_BA_NSA"], color="navy", linewidth=1.2)
    axes[0].set_title("United States: weekly business applications (BA_NSA)")
    axes[0].set_ylabel("Applications")
    axes[0].grid(alpha=0.25)
    tick_step = max(1, len(national) // 12)
    axes[0].set_xticks(range(0, len(national), tick_step))
    axes[0].set_xticklabels(national["Time"].iloc[::tick_step], rotation=45, ha="right")
else:
    axes[0].set_visible(False)

if state_ba_columns and not latest_state_ba.empty:
    selected_states = latest_state_ba.head(5).index.tolist()
    state_trend = combined[["Year", "Week"] + selected_states].copy()
    state_trend["Time"] = state_trend["Year"].astype(str) + "-W" + state_trend["Week"].astype(str).str.zfill(2)
    for column in selected_states:
        axes[1].plot(state_trend["Time"], state_trend[column], linewidth=1, label=column.removesuffix("_BA_NSA"))
    axes[1].set_title("Selected states: weekly business applications (BA_NSA)")
    axes[1].set_ylabel("Applications")
    axes[1].legend(title="State", ncol=min(5, len(selected_states)))
    axes[1].grid(alpha=0.25)
    tick_step = max(1, len(state_trend) // 12)
    axes[1].set_xticks(range(0, len(state_trend), tick_step))
    axes[1].set_xticklabels(state_trend["Time"].iloc[::tick_step], rotation=45, ha="right")
else:
    axes[1].set_visible(False)

plt.show()